# Token Analysis with DadaGP Parser

This notebook analyzes tokens using the `dadagp_parser` module:
- Parse DadaGP token files into events
- Compare input vs output sequences
- Analyze token distributions

In [1]:
import sys
from pathlib import Path
from collections import Counter
import random

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from src.dadagp_parser import (
    parse_dadagp_file_to_events,
    NoteOnEvent, NoteOffEvent, TimeShiftEvent, TabEvent
)

print("✓ Imports successful")

✓ Imports successful


## 1. Find Sample Token Files

In [2]:
# Find all token files
dadagp_dir = Path('../DadaGP-v1.1')
token_files = list(dadagp_dir.rglob('*.tokens.txt'))

print(f"Found {len(token_files):,} token files")

# Sample a few files
if token_files:
    sample_files = random.sample(token_files, min(5, len(token_files)))
    print(f"\nSelected {len(sample_files)} files for analysis:")
    for i, f in enumerate(sample_files, 1):
        print(f"  {i}. {f.relative_to(dadagp_dir)}")
else:
    print("No token files found")
    sample_files = []

Found 26,181 token files

Selected 5 files for analysis:
  1. C/Capi, Alex/Capi, Alex - Yao 6.gp4.tokens.txt
  2. M/Madonna/Madonna - Love Profusion.gp4.tokens.txt
  3. J/Jam/Jam - Jam (Lydian) 2.gp3.tokens.txt
  4. K/Kotiteollisuus/Kotiteollisuus - Hulluutta Ja Humalaa.gp3.tokens.txt
  5. L/Legiao Urbana/Legiao Urbana - Que pais e esse.gp3.tokens.txt


## 2. Parse First Sample File

In [3]:
if sample_files:
    sample_file = sample_files[0]
    print(f"Parsing: {sample_file.name}")
    print("="*60)
    
    # Parse into events
    input_events, output_events = parse_dadagp_file_to_events(str(sample_file))
    
    print(f"\nInput events:  {len(input_events):,} tokens")
    print(f"Output events: {len(output_events):,} tokens")
    print(f"Extra events (TAB): {len(output_events) - len(input_events):,}")
    print(f"Length ratio: {len(output_events)/len(input_events):.2f}x")

Parsing: Capi, Alex - Yao 6.gp4.tokens.txt

Input events:  1,710 tokens
Output events: 2,341 tokens
Extra events (TAB): 631
Length ratio: 1.37x


## 3. Analyze Input Sequence (NOTE_ON/OFF + TIME_SHIFT)

In [4]:
if sample_files and input_events:
    # Count event types
    input_types = Counter([e.type for e in input_events])
    
    print("Input sequence token types:")
    for event_type, count in input_types.most_common():
        print(f"  {event_type:15s}: {count:5,} ({count/len(input_events)*100:5.1f}%)")
    
    # Analyze pitches
    pitches = [e.pitch for e in input_events if isinstance(e, NoteOnEvent)]
    if pitches:
        pitch_counts = Counter(pitches)
        print(f"\nPitch statistics:")
        print(f"  Range: {min(pitches)} - {max(pitches)} (MIDI)")
        print(f"  Unique pitches: {len(pitch_counts)}")
        print(f"  Total notes: {len(pitches)}")
        
        # Show most common pitches
        note_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        print(f"\n  Most common pitches:")
        for pitch, count in pitch_counts.most_common(10):
            octave = (pitch // 12) - 1
            note = note_names[pitch % 12]
            print(f"    {note}{octave} (MIDI {pitch:3d}): {count:3d} notes ({count/len(pitches)*100:5.1f}%)")
    
    # Analyze time shifts
    time_shifts = [e.delta for e in input_events if isinstance(e, TimeShiftEvent)]
    if time_shifts:
        shift_counts = Counter(time_shifts)
        print(f"\nTime shift statistics:")
        print(f"  Total shifts: {len(time_shifts)}")
        print(f"  Range: {min(time_shifts)} - {max(time_shifts)} ticks")
        print(f"  Most common shifts:")
        for shift, count in shift_counts.most_common(10):
            print(f"    {shift:5d} ticks: {count:4d} times ({count/len(time_shifts)*100:5.1f}%)")
    
    # Show first 30 events
    print(f"\nFirst 30 input events:")
    print("-"*60)
    for i, event in enumerate(input_events[:30]):
        if isinstance(event, NoteOnEvent):
            print(f"  {i:3d}: NOTE_ON  pitch={event.pitch}")
        elif isinstance(event, NoteOffEvent):
            print(f"  {i:3d}: NOTE_OFF pitch={event.pitch}")
        elif isinstance(event, TimeShiftEvent):
            print(f"  {i:3d}: TIME_SHIFT delta={event.delta}")

Input sequence token types:
  NOTE_ON        :   631 ( 36.9%)
  NOTE_OFF       :   631 ( 36.9%)
  TIME_SHIFT     :   448 ( 26.2%)

Pitch statistics:
  Range: 40 - 71 (MIDI)
  Unique pitches: 32
  Total notes: 631

  Most common pitches:
    B3 (MIDI  59):  45 notes (  7.1%)
    A#3 (MIDI  58):  44 notes (  7.0%)
    G3 (MIDI  55):  42 notes (  6.7%)
    E3 (MIDI  52):  38 notes (  6.0%)
    F3 (MIDI  53):  35 notes (  5.5%)
    C3 (MIDI  48):  33 notes (  5.2%)
    C4 (MIDI  60):  33 notes (  5.2%)
    B2 (MIDI  47):  32 notes (  5.1%)
    D3 (MIDI  50):  28 notes (  4.4%)
    G#3 (MIDI  56):  27 notes (  4.3%)

Time shift statistics:
  Total shifts: 448
  Range: 120 - 3840 ticks
  Most common shifts:
      240 ticks:  302 times ( 67.4%)
      120 ticks:   70 times ( 15.6%)
      480 ticks:   62 times ( 13.8%)
      960 ticks:   12 times (  2.7%)
      720 ticks:    1 times (  0.2%)
     3840 ticks:    1 times (  0.2%)

First 30 input events:
-------------------------------------------

## 4. Analyze Output Sequence (NOTE_ON/OFF + TIME_SHIFT + TAB)

In [5]:
if sample_files and output_events:
    # Count event types
    output_types = Counter([e.type for e in output_events])
    
    print("Output sequence token types:")
    for event_type, count in output_types.most_common():
        print(f"  {event_type:15s}: {count:5,} ({count/len(output_events)*100:5.1f}%)")
    
    # Analyze TAB tokens
    tab_events = [e for e in output_events if isinstance(e, TabEvent)]
    if tab_events:
        print(f"\nTAB token statistics:")
        print(f"  Total TAB tokens: {len(tab_events)}")
        
        # String distribution
        string_counts = Counter([e.string for e in tab_events])
        print(f"\n  String distribution:")
        for string in sorted(string_counts.keys()):
            count = string_counts[string]
            print(f"    String {string}: {count:4d} notes ({count/len(tab_events)*100:5.1f}%)")
        
        # Fret distribution
        fret_counts = Counter([e.fret for e in tab_events])
        print(f"\n  Fret distribution (top 15):")
        for fret, count in fret_counts.most_common(15):
            print(f"    Fret {fret:2d}: {count:4d} notes ({count/len(tab_events)*100:5.1f}%)")
        
        # Most common string-fret combinations
        combo_counts = Counter([(e.string, e.fret) for e in tab_events])
        print(f"\n  Most common string-fret combinations:")
        for (string, fret), count in combo_counts.most_common(15):
            print(f"    String {string}, Fret {fret:2d}: {count:3d} notes ({count/len(tab_events)*100:5.1f}%)")
    
    # Show first 30 events
    print(f"\nFirst 30 output events:")
    print("-"*60)
    for i, event in enumerate(output_events[:30]):
        if isinstance(event, NoteOnEvent):
            print(f"  {i:3d}: NOTE_ON   pitch={event.pitch}")
        elif isinstance(event, NoteOffEvent):
            print(f"  {i:3d}: NOTE_OFF  pitch={event.pitch}")
        elif isinstance(event, TimeShiftEvent):
            print(f"  {i:3d}: TIME_SHIFT delta={event.delta}")
        elif isinstance(event, TabEvent):
            print(f"  {i:3d}: TAB       string={event.string} fret={event.fret}")

Output sequence token types:
  NOTE_ON        :   631 ( 27.0%)
  TAB            :   631 ( 27.0%)
  NOTE_OFF       :   631 ( 27.0%)
  TIME_SHIFT     :   448 ( 19.1%)

TAB token statistics:
  Total TAB tokens: 631

  String distribution:
    String 1:  152 notes ( 24.1%)
    String 2:  123 notes ( 19.5%)
    String 3:  190 notes ( 30.1%)
    String 4:  102 notes ( 16.2%)
    String 5:   45 notes (  7.1%)
    String 6:   19 notes (  3.0%)

  Fret distribution (top 15):
    Fret  3:   90 notes ( 14.3%)
    Fret  5:   78 notes ( 12.4%)
    Fret  0:   75 notes ( 11.9%)
    Fret  7:   69 notes ( 10.9%)
    Fret  6:   53 notes (  8.4%)
    Fret  4:   48 notes (  7.6%)
    Fret  2:   39 notes (  6.2%)
    Fret  8:   39 notes (  6.2%)
    Fret  9:   33 notes (  5.2%)
    Fret 18:   21 notes (  3.3%)
    Fret 15:   12 notes (  1.9%)
    Fret 16:   11 notes (  1.7%)
    Fret 10:   10 notes (  1.6%)
    Fret 13:    9 notes (  1.4%)
    Fret 17:    9 notes (  1.4%)

  Most common string-fret combina

## 5. Sequence Alignment Analysis

Verify that input and output sequences are properly aligned.

In [6]:
if sample_files and input_events and output_events:
    print("Sequence Alignment Check")
    print("="*60)
    
    # Extract input event types (ignoring TAB)
    input_seq = [(e.type, getattr(e, 'pitch', None) or getattr(e, 'delta', None)) 
                 for e in input_events]
    output_seq_no_tab = [(e.type, getattr(e, 'pitch', None) or getattr(e, 'delta', None)) 
                         for e in output_events if not isinstance(e, TabEvent)]
    
    # Check if they match
    if input_seq == output_seq_no_tab:
        print("✓ Input and output sequences are ALIGNED")
        print("  (Output = Input + TAB tokens)")
    else:
        print("✗ Input and output sequences are MISALIGNED")
        print(f"  Input length: {len(input_seq)}")
        print(f"  Output (no TAB) length: {len(output_seq_no_tab)}")
        
        # Find first mismatch
        for i, (inp, out) in enumerate(zip(input_seq, output_seq_no_tab)):
            if inp != out:
                print(f"  First mismatch at position {i}:")
                print(f"    Input:  {inp}")
                print(f"    Output: {out}")
                break
    
    # Calculate timing alignment
    print(f"\nTiming alignment:")
    input_time = sum([e.delta for e in input_events if isinstance(e, TimeShiftEvent)])
    output_time = sum([e.delta for e in output_events if isinstance(e, TimeShiftEvent)])
    print(f"  Input total time:  {input_time:,} ticks")
    print(f"  Output total time: {output_time:,} ticks")
    if input_time == output_time:
        print("  ✓ Time aligned")
    else:
        print(f"  ✗ Time mismatch: {abs(input_time - output_time)} ticks difference")

Sequence Alignment Check
✓ Input and output sequences are ALIGNED
  (Output = Input + TAB tokens)

Timing alignment:
  Input total time:  126,720 ticks
  Output total time: 126,720 ticks
  ✓ Time aligned


## 6. Multi-File Analysis

Aggregate statistics across multiple files.

In [7]:
if sample_files:
    print("Analyzing multiple files...")
    print("="*60)
    
    # Process all sample files
    all_stats = []
    
    for f in sample_files:
        try:
            inp, out = parse_dadagp_file_to_events(str(f))
            
            tab_count = sum(1 for e in out if isinstance(e, TabEvent))
            note_count = sum(1 for e in inp if isinstance(e, NoteOnEvent))
            
            all_stats.append({
                'file': f.name,
                'input_len': len(inp),
                'output_len': len(out),
                'tab_count': tab_count,
                'note_count': note_count,
                'ratio': len(out) / len(inp) if len(inp) > 0 else 0
            })
        except Exception as e:
            print(f"  Error parsing {f.name}: {e}")
    
    if all_stats:
        print(f"\nSuccessfully parsed {len(all_stats)} files")
        print("\nPer-file statistics:")
        print(f"{'File':<40} {'Input':>8} {'Output':>8} {'TAB':>6} {'Notes':>6} {'Ratio':>6}")
        print("-"*80)
        for s in all_stats:
            print(f"{s['file']:<40} {s['input_len']:>8,} {s['output_len']:>8,} "
                  f"{s['tab_count']:>6,} {s['note_count']:>6,} {s['ratio']:>6.2f}")
        
        # Aggregate stats
        total_input = sum(s['input_len'] for s in all_stats)
        total_output = sum(s['output_len'] for s in all_stats)
        total_tabs = sum(s['tab_count'] for s in all_stats)
        total_notes = sum(s['note_count'] for s in all_stats)
        avg_ratio = sum(s['ratio'] for s in all_stats) / len(all_stats)
        
        print("\nAggregate statistics:")
        print(f"  Total input tokens:  {total_input:,}")
        print(f"  Total output tokens: {total_output:,}")
        print(f"  Total TAB tokens:    {total_tabs:,}")
        print(f"  Total notes:         {total_notes:,}")
        print(f"  Avg output/input ratio: {avg_ratio:.2f}")
        print(f"  Avg TAB/note ratio: {total_tabs/total_notes:.2f}" if total_notes > 0 else "N/A")

Analyzing multiple files...

Successfully parsed 5 files

Per-file statistics:
File                                        Input   Output    TAB  Notes  Ratio
--------------------------------------------------------------------------------
Capi, Alex - Yao 6.gp4.tokens.txt           1,710    2,341    631    631   1.37
Madonna - Love Profusion.gp4.tokens.txt     4,111    5,973  1,862  1,862   1.45
Jam - Jam (Lydian) 2.gp3.tokens.txt         3,215    4,550  1,335  1,335   1.42
Kotiteollisuus - Hulluutta Ja Humalaa.gp3.tokens.txt    7,383   10,541  3,158  3,158   1.43
Legiao Urbana - Que pais e esse.gp3.tokens.txt    5,461    7,817  2,356  2,356   1.43

Aggregate statistics:
  Total input tokens:  21,880
  Total output tokens: 31,222
  Total TAB tokens:    9,342
  Total notes:         9,342
  Avg output/input ratio: 1.42
  Avg TAB/note ratio: 1.00
